# EDA: `fraud-review-queue`, timeboxed to a single day

> **The rules of the timebox.**
> 1. The 5 questions below were written **before** the notebook was opened. No
>    more get added along the way.
> 2. Each question exists because it **feeds a concrete decision downstream**.
>    A figure that answers none of the 5 does not go in.
> 3. **Eight figures at most. One day. Answer and close.**
> 4. The V-columns are **not** studied one by one today (design.md §5.4). If
>    they show up, it is only in Q4, through their null pattern.
>
> Today's risk is not difficulty, it is **seduction**. There are interesting
> patterns to find in the V-columns and they are tempting to chase. Do not.

**The answers are still to be written.** What is here is the questions, the
*why*, and the starting code. The interpretation is the deliverable.


## Setup

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)

# Adjust if ingest.py was run against another path.
TXN = "data/processed/transactions.parquet"
IDY = "data/processed/identity.parquet"

con = duckdb.connect()

# The split boundaries (design.md §4), to mark them on the figures.
TRAIN_END, EMBARGO_END, CALIB_END = 119, 129, 155

con.execute(f"SELECT COUNT(*) AS n, AVG(isFraud) AS fraud_rate FROM read_parquet('{TXN}')").df()


## Q1: where does fraud live along the **amount** axis?

**Why this is THE question.** The magnitude of the whole thesis depends on it.
`V` peaks at `p` around 0.25 by algebra, which is guaranteed, but whether the
**difference in dollars** between ranking by value and ranking by score is
*large* depends on there being **high-amount** fraud. If the IEEE-CIS frauds
are nearly all small, the effect can be trivial.

**The decision it feeds:** it sets a realistic expectation **before** the
single evaluation on test, and looking at it today removes the temptation to
bend parameters on the critical day.


In [ ]:
# TransactionAmt by class (log-x), and fraud rate by amount decile.
df = con.execute(f"""
    SELECT TransactionAmt, isFraud
    FROM read_parquet('{TXN}')
""").df()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(0, np.log10(df.TransactionAmt.max()), 60)
for label, flag in (("legit", 0), ("fraud", 1)):
    ax[0].hist(df.loc[df.isFraud == flag, "TransactionAmt"],
               bins=bins, alpha=.5, density=True, label=label)
ax[0].set_xscale("log")
ax[0].set_xlabel("TransactionAmt")
ax[0].set_ylabel("density")
ax[0].legend()

df["amt_decile"] = pd.qcut(df.TransactionAmt, 10, labels=False, duplicates="drop")
rate = df.groupby("amt_decile").isFraud.mean()
ax[1].bar(rate.index, rate.values)
ax[1].set_xlabel("amount decile")
ax[1].set_ylabel("fraud rate")
plt.tight_layout()

# TODO: does fraud concentrate on small amounts, or is there mass at the high
# end? What fraction of the fraudulent *money* sits in the top amount quartile?


**Answer (still to be written):**

_..._


## Q2: is the **temporal split** viable? Are there enough positives in calibration?

**Why.** The split (train 0-119, embargo 120-129, calib 130-155, test 156 on)
only works if every partition has volume and, critically, if the **calibration
partition has enough positives** to fit an isotonic regression (design.md
§6.3). If the fraud rate collapses over the last days, the calibrator and the
test are left without signal.

**The decision it feeds:** it confirms, or corrects, the boundary days of
`SplitConfig`. If calib holds few positives, Platt over isotonic.


In [ ]:
daily = con.execute(f"""
    SELECT day, COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY day ORDER BY day
""").df()

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(daily.day, daily.n)
ax[0].set_ylabel("transactions per day")
ax[1].plot(daily.day, daily.rate)
ax[1].set_ylabel("fraud rate per day")
ax[1].set_xlabel("day")
for a in ax:
    for d in (TRAIN_END, EMBARGO_END, CALIB_END):
        a.axvline(d, ls="--", c="k", alpha=.4)
plt.tight_layout()

# Positives per partition: the number that decides Platt against isotonic.
con.execute(f"""
    SELECT
      CASE WHEN day <= {TRAIN_END} THEN '1_train'
           WHEN day <= {EMBARGO_END} THEN '2_embargo'
           WHEN day <= {CALIB_END} THEN '3_calib'
           ELSE '4_test' END AS part,
      COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY part ORDER BY part
""").df()


**Answer (still to be written):**

_..._


## Q3: does `D1n = day - D1` behave as a **per-card constant**? (is the UID viable?)

**Why.** The UID is built as `card1 + addr1 + D1n`, with the backward-looking
features on top of it. All of that assumes `D1n`, an approximation of the
card's registration date, is stable within a single card. If it is not, the
entity resolution does not work and needs rethinking.

**The decision it feeds:** it de-risks the feature engineering **now**, with a
cheap check, instead of discovering the problem halfway through it.


In [ ]:
probe = con.execute(f"""
    SELECT
        card1, addr1,
        COUNT(*) AS n,
        COUNT(DISTINCT (day - D1)) AS n_distinct_d1n
    FROM read_parquet('{TXN}')
    WHERE D1 IS NOT NULL AND addr1 IS NOT NULL
    GROUP BY card1, addr1
    HAVING COUNT(*) >= 3
""").df()

# Were D1n perfectly constant per (card1, addr1), n_distinct_d1n == 1.
share_constant = (probe.n_distinct_d1n == 1).mean()
print(f"(card1, addr1) groups with >=3 txns: {len(probe):,}")
print(f"Share with a unique D1n: {share_constant:.1%}")

# TODO: is that high enough to trust the UID? What happens to the tail of
# groups with several D1n values: noise in addr1, or does the proxy break?


**Answer (still to be written):**

_..._


## Q4: identity coverage and null patterns (today's only look at the V-columns)

**Why.** Two feature-engineering decisions come out of here:

1. `train_identity` covers only a fraction of the transactions, so the join is
   a **left join, not an inner one** (design.md §3.2). The exact number matters.
2. The 339 V-columns group into Vesta blocks that **share a null pattern**
   (§5.4). How many blocks there are decides whether to pick one representative
   per block or hand them all to LightGBM whole.

**The decision it feeds:** the identity join strategy and the V-column
strategy. **No column-by-column archaeology.**


In [ ]:
n_txn = con.execute(f"SELECT COUNT(*) FROM read_parquet('{TXN}')").fetchone()[0]
n_idy = con.execute(f"SELECT COUNT(*) FROM read_parquet('{IDY}')").fetchone()[0]
print(f"Identity coverage: {n_idy:,} / {n_txn:,} = {n_idy/n_txn:.1%}  -> left join.")

# Null patterns of the V-columns: how many distinct blocks there are.
vcols = [c for c in con.execute(f"SELECT * FROM read_parquet('{TXN}') LIMIT 0").df().columns
         if c.startswith("V")]
sample = con.execute(
    f"SELECT {', '.join(vcols)} FROM read_parquet('{TXN}') USING SAMPLE 20000 ROWS"
).df()
null_signature = sample.isnull().mean().round(3)          # share of nulls per V-col
n_blocks = null_signature.nunique()
print(f"V-columns: {len(vcols)}  ->  {n_blocks} distinct null patterns (Vesta blocks).")

# TODO: how many blocks? One representative per block, or all of them to LightGBM?


**Answer (still to be written):**

_..._


## Q5: which base categoricals separate fraud? (`ProductCD`, `card4`, `card6`, email)

**Why.** Before investing in features, confirm which low-cardinality
categoricals carry signal, and look at the cardinality of the high ones
(`card1`, `addr1`) for the frequency encoding (design.md §5.1). Cheap and
directly actionable.

**The decision it feeds:** the set of base features, and which categoricals
deserve a frequency encoding computed **on train only**.


In [ ]:
for colname in ["ProductCD", "card4", "card6"]:
    print(f"=== {colname} ===")
    print(con.execute(f"""
        SELECT {colname}, COUNT(*) AS n, AVG(isFraud) AS fraud_rate
        FROM read_parquet('{TXN}')
        GROUP BY {colname} ORDER BY n DESC
    """).df().to_string(index=False))
    print()

# Cardinality of the high-cardinality ones, the frequency-encoding candidates.
print("=== cardinality ===")
print(con.execute(f"""
    SELECT
      COUNT(DISTINCT card1) AS card1,
      COUNT(DISTINCT addr1) AS addr1,
      COUNT(DISTINCT P_emaildomain) AS p_email
    FROM read_parquet('{TXN}')
""").df().to_string(index=False))

# TODO: which categories sit well above the base fraud rate? Which email
# domains get grouped into 'other'?


**Answer (still to be written):**

_..._


## Closing the timebox

- [ ] The 5 questions are answered in prose, not only in figures.
- [ ] At most 8 figures. No figure that answers none of the 5.
- [ ] The split boundary days are confirmed (Q2), or the adjustment is noted.
- [ ] Decided: Platt or isotonic (Q2), UID yes or no (Q3), V-column strategy (Q4).
- [ ] **Closed the notebook. No further exploring of the V-columns.**
